<a href="https://colab.research.google.com/github/Mohamed-hanafy30/AB_testing_Udacity/blob/master/annitia_data_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install lifelines scikit-survival

In [ ]:
import pandas as pd
import numpy as np
import re
from lightgbm import LGBMRegressor
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from lifelines.utils import concordance_index
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
from sklearn.base import clone
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from scipy.stats import rankdata
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/belalemadhussein/annitia/DB-1773398340961.csv')
test = pd.read_csv('/kaggle/input/datasets/belalemadhussein/annitia/DB-1773398340961 (1).csv')
ss = pd.read_csv('/kaggle/input/datasets/belalemadhussein/annitia/hello_world_submission-1773575379610.csv')

In [ ]:
age_cols = [c for c in train.columns if c.startswith('Age_v')]


In [ ]:
def feat_eng(df):

    df = df.copy()

    # detect repeated columns pattern: name_v#
    pattern = re.compile(r"(.*)_v\d+")

    base_cols = {}

    for col in df.columns:
        m = pattern.match(col)
        if m:
            base = m.group(1)
            base_cols.setdefault(base, []).append(col)

    # create aggregation features
    for base, cols in base_cols.items():

        cols_sorted = sorted(cols, key=lambda x: int(x.split("_v")[-1]))
        vals = df[cols_sorted]

        df[f"{base}_mean"] = vals.mean(axis=1)
        df[f"{base}_std"] = vals.std(axis=1)

        df[f"{base}_first"] = vals.iloc[:, 0]
        df[f"{base}_last"] = vals.iloc[:, -1]

        # number of available measurements
        df[f"{base}_count"] = vals.notna().sum(axis=1)

        if base != 'Age':
            df = df.drop(columns = cols_sorted)


    return df

In [ ]:
targets = ['risk_death', 'risk_hepatic_event']

In [ ]:
train['last_observed_age'] = train[age_cols].max(axis=1)
test['last_observed_age'] = test[age_cols].max(axis=1)


In [ ]:
def prepare_survival_targets(df, outcome='hepatic'):

    df = df.copy()

    if outcome == 'hepatic':
        event_col     = 'evenements_hepatiques_majeurs'
        age_occur_col = 'evenements_hepatiques_age_occur'
        name          = 'Hepatic_event'
        is_event = df[event_col] == 1
        invalid  = is_event & df[age_occur_col].isna()
        mask     = ~invalid

    elif outcome == 'death':
        event_col     = 'death'
        age_occur_col = 'death_age_occur'
        name          = 'Death'
        is_event = df[event_col] == 1
        unknown  = df[event_col].isna()
        invalid  = is_event & df[age_occur_col].isna()
        mask     = ~unknown & ~invalid

    else:
        raise ValueError(f"outcome must be 'hepatic' or 'death', got {outcome!r}")

    df_valid   = df[mask].copy().reset_index(drop=True)
    is_event_v = (df_valid[event_col] == 1)

    # Survival time (in same age-year units as the dataset)
    time_values = np.where(
        is_event_v,
        df_valid[age_occur_col] - df_valid['Age_v1'],       # event patients
        df_valid['last_observed_age'] - df_valid['Age_v1'], # censored patients
    ).astype(float)

    # Clamp to small positive value (sksurv requires strictly positive times)
    time_values = np.maximum(time_values, 0.001)

    y = Surv.from_arrays(
        event=is_event_v.astype(bool).values,
        time=time_values,
        name_event=name,
        name_time='Time_years',
    )
    return df_valid, y



In [ ]:
# --- Sanity check ---
df_hep,   y_hep   = prepare_survival_targets(feat_eng(train), outcome='hepatic')
df_death, y_death = prepare_survival_targets(feat_eng(train), outcome='death')


In [ ]:
def build_feature_matrix(df, keep_cols=None):
    """
    Build numeric feature matrix, excluding:
      - Target columns (would cause leakage)
      - ID columns
      - Age_v* columns (used only for censoring time derivation)
      - 'last_observed_age' (derived from Age_v* — leakage risk)
    If keep_cols is provided, only those columns are returned (for
    consistent alignment across train / val / test).
    """
    age_v_cols   = [c for c in df.columns if c.startswith('Age_v')]
    leakage_cols = ['last_observed_age']
    drop_cols    = targets + ['patient_id_anon', 'trustii_id']+ age_v_cols + leakage_cols
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    return X

MAX_MISSING_RATE = 0.8


# ── Build raw matrices ────────────────────────────────────────────────────
X_hep_raw   = build_feature_matrix(df_hep)
X_death_raw = build_feature_matrix(df_death)
X_pred_raw  = build_feature_matrix(feat_eng(test))

print(f'Raw feature count: {X_hep_raw.shape[1]}')

# ── Missing-rate filter (fitted on train, applied everywhere) ─────────────
# Features with > MAX_MISSING_RATE missing in train collapse to a constant
# after median imputation → the model memorises the imputed pattern instead
# of a real signal (a form of overfitting).  Removing them first dramatically
# reduces feature/event ratio (261 → ~36) which is the primary cause of the
# train/val gap.
missing_rate_hep   = X_hep_raw.isna().mean()
missing_rate_death = X_death_raw.isna().mean()

keep_hep   = missing_rate_hep[missing_rate_hep   <= MAX_MISSING_RATE].index.tolist()
keep_death = missing_rate_death[missing_rate_death <= MAX_MISSING_RATE].index.tolist()

# Align to val_test (prediction set must have the same columns)
keep_hep   = [c for c in keep_hep   if c in X_pred_raw.columns if 'Age' not in c]
keep_death = [c for c in keep_death   if c in X_pred_raw.columns]

# Final aligned matrices
X_hep_aln   = X_hep_raw[keep_hep]
X_death_aln = X_death_raw[keep_death]
X_pred_hep  = X_pred_raw[keep_hep]
X_pred_death= X_pred_raw[keep_death]

print(f'Hepatic features after missing filter : {len(keep_hep)}  '
      f'(EPV = {int((df_hep["evenements_hepatiques_majeurs"]==1).sum())}/{len(keep_hep)} = '
      f'{(df_hep["evenements_hepatiques_majeurs"]==1).sum()/len(keep_hep):.2f})')
print(f'Death   features after missing filter : {len(keep_death)}  '
      f'(EPV = {int((df_death["death"]==1).sum())}/{len(keep_death)} = '
      f'{(df_death["death"]==1).sum()/len(keep_death):.2f})')


In [ ]:
# ── Base pipelines ────────────────────────────────────────────────────────
PIPELINES = {
    'rsf': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('m', RandomSurvivalForest(n_estimators=100,  n_jobs=-1, random_state=42)),
    ]),
    'gbm': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('m', GradientBoostingSurvivalAnalysis(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)),
    ]),
    'cox': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('m', CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.1, max_iter=10000)),
    ]),

}

def fit_stacked_ensemble(X, y, event_col, X_pred, n_splits=5):
    names   = list(PIPELINES.keys())
    events  = y[event_col].astype(int)
    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_mat  = np.zeros((len(X),      len(names)))
    test_mat = np.zeros((len(X_pred), len(names)))

    for m_idx, name in enumerate(names):
        print(f'  [{name}]', end='')
        fold_test = np.zeros((len(X_pred), n_splits))
        for fold, (tr, val) in enumerate(skf.split(X, events)):
            m = clone(PIPELINES[name])
            m.fit(X.iloc[tr], y[tr])
            oof_mat[val, m_idx]    = m.predict(X.iloc[val])
            fold_test[:, fold]     = m.predict(X_pred)
            ci = concordance_index_censored(y[val][event_col], y[val]['Time_years'], oof_mat[val, m_idx])[0]
            print(f'  fold{fold+1}={ci:.4f}', end='')
        test_mat[:, m_idx] = fold_test.mean(axis=1)
        oof_ci = concordance_index_censored(y[event_col], y['Time_years'], oof_mat[:, m_idx])[0]
        print(f'  OOF={oof_ci:.4f}')

    # Rank-normalize
    oof_r  = np.column_stack([rankdata(oof_mat[:,i])  for i in range(len(names))])
    test_r = np.column_stack([rankdata(test_mat[:,i]) for i in range(len(names))])

    # Optimize blend weights on OOF C-index
    def neg_ci(w):
        w = np.abs(w) / np.abs(w).sum()
        return -concordance_index_censored(y[event_col], y['Time_years'], oof_r @ w)[0]
    w = np.abs(minimize(neg_ci, x0=np.ones(len(names)), method='Nelder-Mead').x)
    w /= w.sum()
    print('\n  Blend weights:', {n: f'{v:.3f}' for n, v in zip(names, w)})

    oof_final  = oof_r  @ w
    pred_final = test_r @ w
    oof_ci = concordance_index_censored(y[event_col], y['Time_years'], oof_final)[0]
    print(f'  Ensemble OOF C-index: {oof_ci:.4f}')
    return oof_final, pred_final, oof_ci, w

# ── Hepatic ───────────────────────────────────────────────────────────────
print('=== Hepatic ===')
oof_hep, pred_hep, ci_hep, w_hep = fit_stacked_ensemble(
    X_hep_aln, y_hep, event_col='Hepatic_event', X_pred=X_pred_hep
)

# ── Death ─────────────────────────────────────────────────────────────────
print('\n=== Death ===')
oof_death, pred_death, ci_death, w_death = fit_stacked_ensemble(
    X_death_aln, y_death, event_col='Death', X_pred=X_pred_death
)

In [ ]:
# ── Submission ────────────────────────────────────────────────────────────
submission = pd.DataFrame({
    'trustii_id':         test['trustii_id'].values,
    'risk_hepatic_event': pred_hep,
    'risk_death':         pred_death,
})
submission.to_csv('sub.csv', index=False)
print(submission.head())